In [2]:
from datasets import load_dataset
import pandas as pd
import re

# Load full dataset
ds = load_dataset("artem9k/ai-text-detection-pile", split="train")
df = pd.DataFrame(ds)

# Quick quality checks
print(f"Total samples: {len(df)}")
print(f"Class distribution:\n{df['source'].value_counts()}")
print(f"Missing values:\n{df.isnull().sum()}")

Total samples: 1392522
Class distribution:
source
human    1028146
ai        364376
Name: count, dtype: int64
Missing values:
source    0
id        0
text      0
dtype: int64


In [3]:
# Convert to numeric: 1 = AI, 0 = Human
df['label'] = (df['source'] == 'ai').astype(int)

# Verify
print(f"Class Distribution:\n{df['label'].value_counts()}")
print(f"Ratio (human:AI) {1028146 / 364376:.2f}:1")

Class Distribution:
label
0    1028146
1     364376
Name: count, dtype: int64
Ratio (human:AI) 2.82:1


In [4]:
def clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Keep basic punctuation for stylistic features later
    return text.lower()

df['cleaned_text'] = df['text'].apply(clean_text)

In [5]:
from sklearn.model_selection import train_test_split

# 30% stratified sample for faster processing
df_medium = df.sample(frac=0.3, random_state=42)
df_medium = df_medium.reset_index(drop=True)

print(f"df_medium size: {len(df_medium):,} samples")
print(f"Class distribution:\n{df_medium['label'].value_counts( )}")

X = df_medium['cleaned_text']
y = df_medium['label']

# 70-15-15 split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val, = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp # 15% of total
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

df_medium size: 417,757 samples
Class distribution:
label
0    308536
1    109221
Name: count, dtype: int64
Train: 292419, Val: 62674, Test: 62664


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Based on your analysis: focus on distinctive vocabulary
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2), # unigrams + bigrams
    min_df=5, # must appear in at least 5 docs
    max_df=0.8 # ignore terms in >80% of docs
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF shape: {X_train_tfidf.shape}")

TF-IDF shape: (292419, 5000)


In [ ]:
import pickle

# save splits
splits = {
    'X_train': X_train, 'y_train': y_train,
    'X_val': X_val, 'y_val': y_val,
    'X_test': X_test, 'y_test': y_test
}
with open('../data/processed/train_test_splits_medium.pkl', 'wb') as f:
    pickle.dump(splits, f)
    
# Save TF-IDF features
tfidf_data = {
    'X_train': X_train_tfidf,
    'X_val': X_val_tfidf,
    'X_test': X_test_tfidf,
    'vectorizer': tfidf
}
with open('../data/features/tfidf_features_medium.pkl', 'wb') as f:
    pickle.dump(tfidf_data, f)
    
print("Saved successfully")

Saved successfully


In [8]:
import os

print(f"Splits: {os.path.getsize('../data/processed/train_test_splits_mediun.pkl') / 1e6:.1f} MB")
print(f"TF-IDF: {os.path.getsize('../data/features/tfidf_features_medium.pkl') / 1e6:.1f} MB")

Splits: 1009.9 MB
TF-IDF: 918.0 MB
